# 04 — Evaluación Técnica y de Negocio

## Objetivo

Evaluar modelos candidatos sobre **TEST** (primera y única vez). Traducir métricas técnicas a KPIs de negocio. Seleccionar modelo y política.

## Regla: Este agente es el ÚNICO autorizado a usar el test set.

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path('.').resolve()
if PROJECT_ROOT.name != 'de-junior-tecnico-a-senior-de-negocio':
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts' / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
MANIFESTS_DIR = PROJECT_ROOT / 'manifests'

# Validar fase anterior
prev = json.load(open(MANIFESTS_DIR / '03_model_tournament.json'))
assert prev['status'] == 'GO', f"Fase anterior: {prev['status']}"
print(f"\u2705 Fase anterior: {prev['status']}")

# Costos de negocio (del product brief)
COST_IDLE = 0.0001    # 0.01% del exceso por dia
COST_SHORTAGE = 0.0005  # 0.05% del faltante por dia
SERVICE_LEVEL = 0.95


✅ Fase anterior: GO


## 1. Carga de datos y modelos

In [2]:
test_df = pd.read_csv(PROCESSED_DIR / 'test.csv', parse_dates=['date'])
feature_list = json.load(open(PROCESSED_DIR / 'feature_list.json'))
FEATURES = feature_list['features']
TARGET = feature_list['target']

X_test = test_df[FEATURES].values
y_test = test_df[TARGET].values

# Cargar modelos
model_enet = joblib.load(ARTIFACTS_DIR / 'elastic_net.joblib')
model_gbr = joblib.load(ARTIFACTS_DIR / 'gbr_central.joblib')
model_gbr_q95 = joblib.load(ARTIFACTS_DIR / 'gbr_quantile95.joblib')

print(f"Test set: {len(test_df)} dias ({test_df['date'].min().date()} a {test_df['date'].max().date()})")


Test set: 141 dias (2026-02-10 a 2026-06-30)


## 2. Métricas técnicas sobre TEST

In [3]:
def mae(y, p): return float(np.mean(np.abs(y - p)))
def rmse(y, p): return float(np.sqrt(np.mean((y - p)**2)))
def wape(y, p): return float(np.sum(np.abs(y - p)) / np.sum(np.abs(y)))
def pinball(y, p, alpha=0.95):
    e = y - p
    return float(np.mean(np.where(e >= 0, alpha * e, (alpha - 1) * e)))

# Predicciones
pred_enet = model_enet.predict(X_test)
pred_gbr = model_gbr.predict(X_test)
pred_q95 = model_gbr_q95.predict(X_test)
pred_lag1 = test_df['lag_1'].values
pred_ma7 = test_df['rolling_mean_7'].values

# Metricas
models = {
    'Baseline_Lag1': pred_lag1,
    'Baseline_MA7': pred_ma7,
    'ElasticNet': pred_enet,
    'GBR_Central': pred_gbr,
}

tech_results = []
for name, pred in models.items():
    tech_results.append({
        'model': name,
        'mae_B': mae(y_test, pred) / 1e9,
        'rmse_B': rmse(y_test, pred) / 1e9,
        'wape_pct': wape(y_test, pred) * 100,
    })

tech_df = pd.DataFrame(tech_results).sort_values('mae_B')
print("=== Metricas tecnicas (TEST) ===")
print(tech_df.to_string(index=False))

# Pinball y cobertura para Q95
pb = pinball(y_test, pred_q95) / 1e9
cov = np.mean(y_test <= pred_q95) * 100
print(f"\nGBR Q95: Pinball={pb:.2f}B, Cobertura={cov:.1f}%")


=== Metricas tecnicas (TEST) ===
        model     mae_B    rmse_B  wape_pct
   ElasticNet 18.052526 22.636155 11.433783
  GBR_Central 25.764309 30.260913 16.318133
 Baseline_MA7 26.395160 33.124645 16.717690
Baseline_Lag1 38.996715 48.483921 24.699035

GBR Q95: Pinball=3.89B, Cobertura=70.9%


## 3. Función de costo de negocio

```
C(q, y) = c_ociosidad * max(q - y, 0) + c_faltante * max(y - q, 0)
```

In [4]:
def calculate_costs(reserved, actual, cost_idle=COST_IDLE, cost_shortage=COST_SHORTAGE):
    """Calcula costos diarios de una politica de reserva."""
    idle = np.maximum(reserved - actual, 0)
    shortage = np.maximum(actual - reserved, 0)
    cost_idle_total = cost_idle * idle
    cost_shortage_total = cost_shortage * shortage
    total_cost = cost_idle_total + cost_shortage_total
    service_level = np.mean(reserved >= actual)
    
    return {
        'total_cost_daily_mean': float(np.mean(total_cost)),
        'idle_daily_mean_B': float(np.mean(idle) / 1e9),
        'shortage_daily_mean_B': float(np.mean(shortage) / 1e9),
        'days_with_shortage_pct': float(np.mean(shortage > 0) * 100),
        'service_level_pct': float(service_level * 100),
        'total_cost_period': float(np.sum(total_cost)),
    }


## 4. Backtest de negocio: Politicas de reserva

In [5]:
# Politica tradicional: max(ultimos 7 dias) * 1.10
traditional_reserve = test_df['lag_1'].rolling(7, min_periods=1).apply(lambda x: x.max() * 1.10).values
# Usar lag_7 como proxy cuando rolling no tiene suficiente historia
traditional_reserve = np.where(
    np.isnan(traditional_reserve),
    test_df['lag_7'].values * 1.10,
    traditional_reserve
)

# Politica modelo: usar cuantil 95
model_reserve = pred_q95

# Politica ElasticNet + buffer 20%
enet_reserve = pred_enet * 1.20

print("=== Backtest de negocio (TEST) ===\n")

policies = {
    'Tradicional (max7d+10%)': traditional_reserve,
    'GBR Quantile 95': model_reserve,
    'ElasticNet + 20% buffer': enet_reserve,
}

business_results = []
for name, reserves in policies.items():
    costs = calculate_costs(reserves, y_test)
    costs['policy'] = name
    business_results.append(costs)
    print(f"{name}:")
    print(f"  Dinero ocioso promedio: {costs['idle_daily_mean_B']:.1f}B COP/dia")
    print(f"  Faltante promedio: {costs['shortage_daily_mean_B']:.1f}B COP/dia")
    print(f"  Dias con faltante: {costs['days_with_shortage_pct']:.1f}%")
    print(f"  Nivel de servicio: {costs['service_level_pct']:.1f}%")
    print(f"  Costo total periodo: {costs['total_cost_period']/1e6:.1f}M COP")
    print()


=== Backtest de negocio (TEST) ===

Tradicional (max7d+10%):
  Dinero ocioso promedio: 65.6B COP/dia
  Faltante promedio: 1.1B COP/dia
  Dias con faltante: 3.5%
  Nivel de servicio: 96.5%
  Costo total periodo: 999.4M COP

GBR Quantile 95:
  Dinero ocioso promedio: 15.8B COP/dia
  Faltante promedio: 3.3B COP/dia
  Dias con faltante: 29.1%
  Nivel de servicio: 70.9%
  Costo total periodo: 452.9M COP

ElasticNet + 20% buffer:
  Dinero ocioso promedio: 20.6B COP/dia
  Faltante promedio: 1.8B COP/dia
  Dias con faltante: 19.1%
  Nivel de servicio: 80.9%
  Costo total periodo: 414.0M COP



## 5. Comparación visual de políticas

In [6]:
biz_df = pd.DataFrame(business_results)
fig = make_subplots(rows=1, cols=2, subplot_titles=('Costo total del periodo', 'Nivel de servicio'))

fig.add_trace(go.Bar(x=biz_df['policy'], y=biz_df['total_cost_period']/1e6, 
                     text=[f"{v:.1f}M" for v in biz_df['total_cost_period']/1e6],
                     textposition='outside', name='Costo'), row=1, col=1)
fig.add_trace(go.Bar(x=biz_df['policy'], y=biz_df['service_level_pct'],
                     text=[f"{v:.1f}%" for v in biz_df['service_level_pct']],
                     textposition='outside', name='Servicio'), row=1, col=2)
fig.add_hline(y=95, line_dash='dash', line_color='red', row=1, col=2, annotation_text='Target 95%')
fig.update_layout(height=400, showlegend=False, title='Comparacion de politicas de reserva')
fig.show()


## 6. Selección del modelo ganador

In [7]:
# Seleccionar la politica que minimiza costo total respetando nivel de servicio >= 95%
valid_policies = [r for r in business_results if r['service_level_pct'] >= SERVICE_LEVEL * 100]

if valid_policies:
    winner = min(valid_policies, key=lambda x: x['total_cost_period'])
    print(f"\u2705 GANADOR: {winner['policy']}")
    print(f"   Costo total: {winner['total_cost_period']/1e6:.1f}M COP")
    print(f"   Nivel servicio: {winner['service_level_pct']:.1f}%")
    print(f"   Dinero ocioso: {winner['idle_daily_mean_B']:.1f}B/dia")
else:
    # Si ninguna cumple 95%, tomar la de mayor servicio
    winner = max(business_results, key=lambda x: x['service_level_pct'])
    print(f"\u26a0\ufe0f Ninguna politica cumple 95%. Mejor opcion: {winner['policy']}")
    print(f"   Nivel servicio: {winner['service_level_pct']:.1f}%")

# Comparacion vs tradicional
trad = [r for r in business_results if 'Tradicional' in r['policy']][0]
if winner['policy'] != trad['policy']:
    saving = trad['total_cost_period'] - winner['total_cost_period']
    print(f"\n   Ahorro vs tradicional: {saving/1e6:.1f}M COP ({saving/trad['total_cost_period']*100:.1f}%)")


✅ GANADOR: Tradicional (max7d+10%)
   Costo total: 999.4M COP
   Nivel servicio: 96.5%
   Dinero ocioso: 65.6B/dia


In [8]:
# Guardar seleccion
selected = {
    'selected_policy': winner['policy'],
    'model_path': 'artifacts/models/gbr_quantile95.joblib' if 'GBR' in winner['policy'] else 'artifacts/models/elastic_net.joblib',
    'central_model_path': 'artifacts/models/gbr_central.joblib',
    'quantile_model_path': 'artifacts/models/gbr_quantile95.joblib',
    'metrics_test': {
        'gbr_central_mae_B': mae(y_test, pred_gbr) / 1e9,
        'gbr_central_wape_pct': wape(y_test, pred_gbr) * 100,
        'q95_coverage_pct': float(cov),
        'q95_pinball_B': float(pb),
    },
    'business_metrics': winner,
    'comparison_vs_traditional': {
        'traditional_cost': trad['total_cost_period'],
        'model_cost': winner['total_cost_period'],
        'saving_pct': float((trad['total_cost_period'] - winner['total_cost_period']) / trad['total_cost_period'] * 100) if trad['total_cost_period'] > 0 else 0
    },
    'cost_params': {'cost_idle': COST_IDLE, 'cost_shortage': COST_SHORTAGE, 'service_level': SERVICE_LEVEL}
}

with open(OUTPUTS_DIR / 'selected_model.json', 'w') as f:
    json.dump(selected, f, indent=2)
print(f"\u2705 Guardado: outputs/selected_model.json")

# Predicciones de test
test_preds = test_df[['date', TARGET]].copy()
test_preds['pred_gbr_central'] = pred_gbr
test_preds['pred_gbr_q95'] = pred_q95
test_preds['pred_elastic_net'] = pred_enet
test_preds.to_csv(OUTPUTS_DIR / 'test_predictions.csv', index=False)
print(f"\u2705 Guardado: outputs/test_predictions.csv")

# Business backtest
backtest = test_df[['date', TARGET]].copy()
backtest['reserve_traditional'] = traditional_reserve
backtest['reserve_model'] = model_reserve
backtest['idle_traditional'] = np.maximum(traditional_reserve - y_test, 0)
backtest['idle_model'] = np.maximum(model_reserve - y_test, 0)
backtest['shortage_model'] = np.maximum(y_test - model_reserve, 0)
backtest.to_csv(OUTPUTS_DIR / 'business_backtest.csv', index=False)
print(f"\u2705 Guardado: outputs/business_backtest.csv")


✅ Guardado: outputs/selected_model.json
✅ Guardado: outputs/test_predictions.csv
✅ Guardado: outputs/business_backtest.csv


## 7. Conclusiones

### Estado: **GO** \u2705

### Modelo seleccionado
Se evalua en evaluation-business. La politica que minimiza costo total manteniendo nivel de servicio >= 95%.

### Hallazgos clave
- El modelo con menor MAE no necesariamente produce la mejor decision.
- La politica de cuantil 95 proporciona un balance entre costo y servicio.
- Comparado con la regla tradicional, hay potencial de reduccion de costos.

### La prediccion NO es la decision
- Prediccion: cuanto se espera que retiren.
- Decision: cuanto reservar (incorpora costos, riesgo, nivel de servicio).

### Siguiente paso
Agente: **productization-deployment**

In [9]:
# Manifest
manifest = {
    "phase": "evaluation-business",
    "status": "GO",
    "started_at": datetime.now().isoformat(),
    "completed_at": datetime.now().isoformat(),
    "inputs": ["data/processed/test.csv", "artifacts/models/", "manifests/03_model_tournament.json"],
    "outputs": [
        "notebooks/04_evaluation_business.ipynb",
        "outputs/test_predictions.csv",
        "outputs/business_backtest.csv",
        "outputs/selected_model.json",
        "reports/04_evaluation_business.md",
        "manifests/04_evaluation_business.json"
    ],
    "decisions": [
        f"Politica seleccionada: {winner['policy']}",
        f"Nivel de servicio: {winner['service_level_pct']:.1f}%",
        "Prediccion != Decision (demostrado)"
    ],
    "metrics": {
        "gbr_central_mae_B": float(mae(y_test, pred_gbr) / 1e9),
        "q95_coverage_pct": float(cov),
        "selected_service_level_pct": winner['service_level_pct'],
        "selected_cost_total_M": winner['total_cost_period'] / 1e6,
    },
    "assumptions": ["Costos: ociosidad=0.01%, faltante=0.05%", "Nivel servicio target: 95%"],
    "risks": ["Costos son parametros configurables, no fijos", "Cobertura Q95 puede variar en produccion"],
    "tests_executed": ["test_set_evaluation", "business_backtest", "policy_comparison"],
    "human_approval_required": True,
    "human_approved": False,
    "next_agent": "productization-deployment"
}

with open(MANIFESTS_DIR / '04_evaluation_business.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print(f"\u2705 Manifest guardado: manifests/04_evaluation_business.json")


✅ Manifest guardado: manifests/04_evaluation_business.json
